# SPID v4 — Convergence Rate Estimation ($\hat{\rho}$)

Fits the linear convergence constant $\hat{\rho}$ from the tail of the
log-residual sequence $\|X_{k+1} - X_k\|_F$ for Higham, Anderson, and SPID v4.

Populates **Table 7** in the paper, quantifying the "steepest descent in the tail"
claim from §6.4.4.

In [ ]:
import numpy as np
import scipy.linalg as la
import pandas as pd
import time

## 1 — Core Operators

In [ ]:
def proj_U(A):
    """P_U: set diagonal to 1."""
    X = A.copy()
    np.fill_diagonal(X, 1.0)
    return X


def proj_S(A):
    """P_S: clip negative eigenvalues to zero."""
    try:
        w, V = la.eigh(A, subset_by_value=(0.0, np.inf))
        X = V @ np.diag(w) @ V.T
    except ValueError:
        X = np.zeros_like(A)
    return (X + X.T) / 2.0

## 2 — Solvers with Residual Logging

In [ ]:
DENOM_GUARD = 1e-12


def spid_v4_residuals(A, beta=0.5, lambda_max=5.0, alpha_min=0.1,
                      tol=1e-5, max_iter=2000):
    """
    SPID v4 returning (X_hat, iters, converged, residuals).
    residuals[k] = ||X_{k+1} - X_k||_F.
    """
    n = A.shape[0]
    large_n = (n >= 500)
    X = A.copy()
    dS = np.zeros((n, n))
    dU = np.zeros((n, n))

    alpha_max = lambda_max / beta
    alpha_fb = 1.0 / beta

    X_prev = X.copy()
    R_prev = np.zeros((n, n))

    residuals = []
    converged = False

    for k in range(1, max_iter + 1):
        # [1] Dykstra cycle
        R = X + dS
        S = proj_S(R)
        dS_new = R - S
        R_U = S + dU
        W = proj_U(R_U)
        dU_new = R_U - W
        T_D = W

        # [2] Ishikawa nesting
        Z_X  = (1 - beta) * X  + beta * T_D
        Z_dS = (1 - beta) * dS + beta * dS_new
        Z_dU = (1 - beta) * dU + beta * dU_new
        R_curr = X - Z_X

        # [3] BB step
        if k == 1:
            alpha = alpha_fb
        else:
            s = X - X_prev
            y = R_curr - R_prev
            tr_SS = np.sum(s * s)
            tr_SY = np.sum(s * y)
            tr_YY = np.sum(y * y)

            if k % 2 == 1:  # BB1
                alpha = tr_SS / tr_SY if abs(tr_SY) > DENOM_GUARD else alpha_fb
            else:           # BB2
                alpha = tr_SY / tr_YY if abs(tr_YY) > DENOM_GUARD else alpha_fb

            alpha = float(np.clip(alpha, alpha_min, alpha_max))

        # [4] Update
        X_next  = X  + alpha * (Z_X  - X)
        dS_next = dS + alpha * (Z_dS - dS)
        dU_next = dU + alpha * (Z_dU - dU)

        # Record residual
        step_norm = la.norm(X_next - X, 'fro')
        residuals.append(step_norm)

        # [5] Stop
        if large_n:
            stop = step_norm < tol
        else:
            stop = max(la.norm(X_next - S, 'fro'),
                       np.max(np.abs(np.diag(X_next) - 1.0))) < tol

        if stop:
            X = X_next
            converged = True
            break

        # [6] Shift
        X_prev = X
        R_prev = R_curr
        X, dS, dU = X_next, dS_next, dU_next

    X_hat = proj_U(proj_S(X))
    return X_hat, k, converged, residuals


def higham_residuals(A, tol=1e-5, max_iter=2000):
    """
    Higham Dykstra returning (X_hat, iters, converged, residuals).
    """
    n = A.shape[0]
    large_n = (n >= 500)
    X = A.copy()
    dS = np.zeros_like(A)
    dU = np.zeros_like(A)

    residuals = []
    converged = False

    for k in range(1, max_iter + 1):
        R = X + dS
        S = proj_S(R)
        dS_new = R - S
        R_U = S + dU
        W = proj_U(R_U)
        dU_new = R_U - W

        step_norm = la.norm(W - X, 'fro')
        residuals.append(step_norm)

        if large_n:
            stop = step_norm < tol
        else:
            stop = max(la.norm(W - S, 'fro'),
                       np.max(np.abs(np.diag(W) - 1.0))) < tol

        if stop:
            converged = True
            X = W
            break

        X, dS, dU = W, dS_new, dU_new

    X_hat = proj_U(proj_S(X))
    return X_hat, k, converged, residuals


def anderson_residuals(A, m=3, tol=1e-5, max_iter=2000):
    """
    Anderson-accelerated Dykstra (m=3) returning
    (X_hat, iters, converged, residuals).
    """
    n = A.shape[0]
    large_n = (n >= 500)
    X = A.copy()
    dS = np.zeros_like(A)
    dU = np.zeros_like(A)

    # History buffers for Anderson mixing
    X_hist = []
    G_hist = []  # G_k = T_D(X_k) - X_k (fixed-point residual)

    residuals = []
    converged = False

    for k in range(1, max_iter + 1):
        # Dykstra cycle
        R = X + dS
        S = proj_S(R)
        dS_new = R - S
        R_U = S + dU
        W = proj_U(R_U)
        dU_new = R_U - W

        step_norm = la.norm(W - X, 'fro')
        residuals.append(step_norm)

        # Anderson mixing
        gk = (W - X).ravel()
        X_hist.append(X.ravel().copy())
        G_hist.append(gk.copy())

        if len(G_hist) > m + 1:
            X_hist.pop(0)
            G_hist.pop(0)

        mk = len(G_hist) - 1
        if mk >= 1:
            # Build delta matrices
            dG = np.column_stack([G_hist[j+1] - G_hist[j] for j in range(mk)])
            dX = np.column_stack([X_hist[j+1] - X_hist[j] for j in range(mk)])

            # Solve least-squares: min ||gk - dG @ gamma||
            try:
                gamma, _, _, _ = la.lstsq(dG, gk)
                x_aa = (X_hist[-1] + gk) - (dX + dG) @ gamma
                X_next = x_aa.reshape(n, n)
                X_next = (X_next + X_next.T) / 2.0
            except la.LinAlgError:
                X_next = W
        else:
            X_next = W

        # Stop check
        if large_n:
            stop = step_norm < tol
        else:
            stop = max(la.norm(W - S, 'fro'),
                       np.max(np.abs(np.diag(W) - 1.0))) < tol

        if stop:
            converged = True
            X = W
            break

        X, dS, dU = X_next, dS_new, dU_new

    X_hat = proj_U(proj_S(X))
    return X_hat, k, converged, residuals

## 3 — Convergence Rate Estimator

In [ ]:
def estimate_rho(residuals, tail_fraction=0.5):
    """
    Estimate per-step linear convergence rate from the tail
    of the residual sequence.

    In the linear convergence regime:
        ||X_{k+1} - X_k|| ~ C * rho^k
    so log(||X_{k+1} - X_k||) ~ k*log(rho) + log(C).

    We fit a line to the last tail_fraction of the log-residuals
    and return rho = exp(slope).

    Returns (rho, r_squared, k_start, k_end).
    """
    res = np.array([r for r in residuals if r > 0])
    if len(res) < 6:
        return np.nan, np.nan, 0, 0

    n = len(res)
    start = int(n * (1 - tail_fraction))
    start = max(start, 2)  # need at least a few points

    log_res = np.log(res[start:])
    ks = np.arange(len(log_res), dtype=float)

    if len(ks) < 3:
        return np.nan, np.nan, start, n - 1

    # Linear fit
    coeffs = np.polyfit(ks, log_res, 1)
    slope = coeffs[0]
    rho = np.exp(slope)

    # R-squared
    fitted = np.polyval(coeffs, ks)
    ss_res = np.sum((log_res - fitted) ** 2)
    ss_tot = np.sum((log_res - np.mean(log_res)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    return rho, r2, start, n - 1

## 4 — Test Matrices

In [ ]:
def gen_qi_sun_55(n=500, seed=42):
    np.random.seed(seed)
    G = np.random.randn(n, n); C = G @ G.T
    d = np.diag(1.0 / np.sqrt(np.diag(C))); C = d @ C @ d
    R = np.random.uniform(-1, 1, (n, n)); R = (R + R.T) / 2.0
    return C + 1.0 * R

def gen_qi_sun_56(n=500, seed=42):
    np.random.seed(seed)
    G = np.random.uniform(-1, 1, (n, n)); G = (G + G.T) / 2.0
    np.fill_diagonal(G, 1.0)
    return G

def gen_higham_strabic(n=500):
    G = np.ones((n, n)) * 0.5; np.fill_diagonal(G, 1.0)
    G[0, n-1] = G[n-1, 0] = -0.9
    return G

def gen_grubisic_pietersz(n=100, seed=42):
    np.random.seed(seed); r = 5
    F = np.random.randn(n, r); G = F @ F.T
    d = np.diag(1.0 / np.sqrt(np.diag(G))); G = d @ G @ d
    G += 0.05 * np.random.randn(n, n)
    return proj_U((G + G.T) / 2.0)

def gen_russell_3000(n=1000, seed=42):
    np.random.seed(seed)
    G = np.random.randn(n, 10) @ np.random.randn(10, n)
    G = (G + G.T) / 2.0
    return proj_U(G)

def gen_thesis_mcar(n_stocks=550, n_days=1000, missing_rate=0.25, seed=42):
    np.random.seed(seed)
    factors  = np.random.randn(10, n_days)
    loadings = np.random.randn(n_stocks, 10)
    returns  = loadings @ factors + np.random.randn(n_stocks, n_days) * 0.5
    mask     = np.random.rand(n_stocks, n_days) > missing_rate
    observed = np.where(mask, returns, np.nan)
    df       = pd.DataFrame(observed.T)
    G        = df.corr().fillna(0).values
    G        = (G + G.T) / 2.0
    np.fill_diagonal(G, 1.0)
    return G


# Six key problems for Table 7
RHO_TESTS = [
    ("QS5.5 n500",       gen_qi_sun_55()),
    ("QS5.6 n500",       gen_qi_sun_56()),
    ("H-S n500",         gen_higham_strabic()),
    ("Grubisic n100",    gen_grubisic_pietersz()),
    ("MCAR n550",        gen_thesis_mcar()),
    ("Russell n1000",    gen_russell_3000()),
]

print("Test matrices ready:")
for name, G in RHO_TESTS:
    lmin = float(np.min(la.eigvalsh(G)))
    print(f"  {name:<20} n={G.shape[0]:>4}  lam_min={lmin:+.4e}")

## 5 — Run All Three Solvers and Estimate $\hat{\rho}$

In [ ]:
rows = []

for name, G in RHO_TESTS:
    print(f"\n{'='*70}")
    print(f"  {name}  (n={G.shape[0]})")
    print(f"{'='*70}")

    for method_name, solver_fn in [
        ("Higham",   lambda A: higham_residuals(A)),
        ("Anderson", lambda A: anderson_residuals(A, m=3)),
        ("SPID v4",  lambda A: spid_v4_residuals(A)),
    ]:
        t0 = time.perf_counter()
        X_hat, iters, conv, res = solver_fn(G)
        wall = time.perf_counter() - t0

        rho, r2, k_start, k_end = estimate_rho(res)

        tag = f"{iters}" if conv else "DNF"
        rho_str = f"{rho:.4f}" if not np.isnan(rho) else "---"
        r2_str  = f"{r2:.3f}" if not np.isnan(r2) else "---"

        print(f"  {method_name:<10}  iters={tag:>5}  "
              f"rho={rho_str:>7}  R2={r2_str:>6}  "
              f"tail=[{k_start},{k_end}]  ({wall:.1f}s)")

        rows.append({
            "Problem":  name,
            "Method":   method_name,
            "n":        G.shape[0],
            "Iters":    iters,
            "Converged": conv,
            "Rho":      rho,
            "R2":       r2,
            "Tail":     f"[{k_start},{k_end}]",
            "Wall_s":   wall,
        })

df = pd.DataFrame(rows)
print("\nDone.")

## 6 — Summary Table

In [ ]:
print("\n" + "=" * 85)
print("CONVERGENCE RATE SUMMARY")
print("=" * 85)
print(f"{'Problem':<20} | {'rho_H':>7} {'rho_A':>7} {'rho_SP':>7} | "
      f"{'H/SP':>6} {'A/SP':>6} | {'it_H':>5} {'it_A':>5} {'it_SP':>5}")
print("-" * 85)

for name, _ in RHO_TESTS:
    sub = df[df["Problem"] == name]
    rh = sub[sub["Method"]=="Higham"]["Rho"].values[0]
    ra = sub[sub["Method"]=="Anderson"]["Rho"].values[0]
    rs = sub[sub["Method"]=="SPID v4"]["Rho"].values[0]

    ih = sub[sub["Method"]=="Higham"]["Iters"].values[0]
    ia = sub[sub["Method"]=="Anderson"]["Iters"].values[0]
    is_ = sub[sub["Method"]=="SPID v4"]["Iters"].values[0]

    ch = sub[sub["Method"]=="Higham"]["Converged"].values[0]
    ca = sub[sub["Method"]=="Anderson"]["Converged"].values[0]
    cs = sub[sub["Method"]=="SPID v4"]["Converged"].values[0]

    def fmt_rho(r, c):
        if not c: return "  DNF"
        return f"{r:.4f}" if not np.isnan(r) else "  ---"

    def fmt_it(i, c):
        return "  DNF" if not c else f"{i:>5}"

    def fmt_ratio(r1, r2, c1, c2):
        if not c1 or not c2 or np.isnan(r1) or np.isnan(r2) or r2 == 0:
            return "  ---"
        return f"{r1/r2:>5.2f}x"

    print(f"{name:<20} | {fmt_rho(rh,ch):>7} {fmt_rho(ra,ca):>7} {fmt_rho(rs,cs):>7} | "
          f"{fmt_ratio(rh,rs,ch,cs):>6} {fmt_ratio(ra,rs,ca,cs):>6} | "
          f"{fmt_it(ih,ch):>5} {fmt_it(ia,ca):>5} {fmt_it(is_,cs):>5}")

print("=" * 85)
print("\nrho < 1 confirms linear convergence.  Lower rho = faster convergence.")
print("H/SP = ratio of Higham rho to SPID rho (>1 means SPID is faster).")

## 7 — Generate LaTeX Table

In [ ]:
latex_names = {
    "QS5.5 n500":    r"Qi--Sun 5.5 ($n=500$)",
    "QS5.6 n500":    r"Qi--Sun 5.6 ($n=500$)",
    "H-S n500":      r"Higham--Strabic ($n=500$)",
    "Grubisic n100": r"Grubisic--Pietersz ($n=100$)",
    "MCAR n550":     r"MCAR Thesis ($n=550$)",
    "Russell n1000":  r"Russell 3000 ($n=1000$)",
}

lines = [
    r"\begin{table}[ht]",
    r"\centering",
    r"\caption{Estimated linear convergence rate $\hat\rho$ fitted from",
    r"  the tail 50\% of the step-residual sequence",
    r"  $\|X_{k+1}-X_k\|_F$.  $R^2$ measures goodness of",
    r"  the log-linear fit.  Lower $\hat\rho$ is better;",
    r"  $\hat\rho < 1$ confirms linear convergence.",
    r"  Blank entries indicate DNF within 2000~iterations.}",
    r"\label{tab:rho}",
    r"\begin{tabular}{lcccccc}",
    r"\toprule",
    r" & \multicolumn{2}{c}{Higham~\cite{higham2002}}",
    r" & \multicolumn{2}{c}{Anderson~\cite{higham2016}}",
    r" & \multicolumn{2}{c}{SPID v4} \\",
    r"\cmidrule(lr){2-3} \cmidrule(lr){4-5} \cmidrule(lr){6-7}",
    r"Problem & $\hat\rho$ & $R^2$",
    r" & $\hat\rho$ & $R^2$",
    r" & $\hat\rho$ & $R^2$ \\",
    r"\midrule",
]

for name, _ in RHO_TESTS:
    sub = df[df["Problem"] == name]
    lname = latex_names.get(name, name)
    cells = [lname]

    for method in ["Higham", "Anderson", "SPID v4"]:
        row = sub[sub["Method"] == method].iloc[0]
        if not row["Converged"] or np.isnan(row["Rho"]):
            cells.append("---")
            cells.append("---")
        else:
            cells.append(f"{row['Rho']:.4f}")
            r2 = row['R2']
            cells.append(f"{r2:.3f}" if not np.isnan(r2) else "---")

    lines.append(" & ".join(cells) + r" \\")

lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]

latex_str = "\n".join(lines)
print("\n" + "=" * 70)
print("LaTeX TABLE (paste into paper as Table 7):")
print("=" * 70)
print(latex_str)